In [ ]:
#!/usr/bin/env python
import os 
impory sys
#import pyhanami
import xarray as xr


In [5]:
final_path = '/scratch/b/b382291/caribbean/storylines/cont/member_1/psl_uas_vas_z_2018.nc'
folder_path = '/scratch/b/b382291/caribbean/storylines/cont/member_1/plots'

In [ ]:
os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]

In [3]:
# running pyhanami 

g0 = 9.80665

# ds_uas = xr.open_mfdataset('/gpfs/projects/bsc32/HANAMI/data/destine/IFS-NEMO-5km-o007-hist/uas/uas-??????-6hourly-r100-hist-5km.nc')
# ds_vas = xr.open_mfdataset('/gpfs/projects/bsc32/HANAMI/data/destine/IFS-NEMO-5km-o007-hist/vas/vas-??????-6hourly-r100-hist-5km.nc')
# ds_psl = xr.open_mfdataset('/gpfs/projects/bsc32/HANAMI/data/destine/IFS-NEMO-5km-o007-hist/psl/psl-??????-6hourly-r100-hist-5km.nc')
# ds_z300 = xr.open_mfdataset('/gpfs/projects/bsc32/HANAMI/data/destine/IFS-NEMO-5km-o007-hist/z300/z300-??????-6hourly-r100-hist-5km.nc')
# ds_z500 = xr.open_mfdataset('/gpfs/projects/bsc32/HANAMI/data/destine/IFS-NEMO-5km-o007-hist/z500/z500-??????-6hourly-r100-hist-5km.nc')



# ds_zg300_5km = ds_z300.rename({'z300':'zg300'})
# ds_zg300_5km['zg300'] = ds_zg300_5km['zg300']/g0

# ds_zg500_5km = ds_z500.rename({'z500':'zg500'})
# ds_zg500_5km['zg500'] = ds_zg500_5km['zg500']/g0
# ds_zg300_5km.zg300.attrs['long_name'] = 'Geopotential height at 300 hPa'
# ds_zg300_5km.zg300.attrs['units'] = 'm'
# ds_zg500_5km.zg500.attrs['long_name'] = 'Geopotential height at 500 hPa'
# ds_zg500_5km.zg500.attrs['units'] = 'm'
# #
# ds_TC_5km = xr.merge([ds_uas, ds_vas, ds_psl, ds_zg300_5km, ds_zg500_5km])
# ds_TC_5km.to_netcdf('/gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/data/tropical_cyclones/TC_data_tco2559_1x1.nc')


# above script is here on MN5 
# /gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/data/tropical_cyclones/TC_data_tco2559_1x1.nc
import pandas as pd 

years = pd.date_range(start="2018", end="2025", freq='YE')
years = years.strftime('%Y').tolist()

exps = ["cont", "hist", "Tplus2.0K"]
for exp in exps:
    for year in years:
        ds_msl = xr.open_mfdataset(f'/scratch/b/b382291/caribbean/storylines/{exp}/member_1/annual/msl_u10_v10_{year}_01_01_to_{year}_12_31.nc')
        ds_z = xr.open_mfdataset(f'/scratch/b/b382291/caribbean/storylines/{exp}/member_1/annual/z_{year}_01_01_to_{year}_12_31.nc')

        final_path = f'/scratch/b/b382291/caribbean/storylines/{exp}/member_1/psl_uas_vas_z_{year}.nc'

        # Check the level dimension name and values first
        print(ds_z.z.dims, ds_z.z.attrs.get('units'))
        lev_dim = [d for d in ds_z.z.dims if d not in ('time', 'lat', 'lon', 'latitude', 'longitude')][0]
        print(lev_dim, ds_z[lev_dim].values)

        def to_latlon(ds):
            mapping = {k: v for k, v in {'latitude': 'lat', 'longitude': 'lon'}.items() if k in ds.dims or k in ds.coords}
            return ds.rename(mapping)

        ds_msl = to_latlon(ds_msl)
        ds_z = to_latlon(ds_z)


        # Geopotential -> geopotential height
        zg = ds_z['z'] / g0

        # One DataArray per level: zg300, zg500, ...
        zg_vars = {}
        for lev in zg[lev_dim].values:
            name = f"zg{int(lev)}"
            da = zg.sel({lev_dim: lev}, drop=True)
            da.attrs = {
                'long_name': f'Geopotential height at {int(lev)} hPa',
                'standard_name': 'geopotential_height',
                'units': 'm',
            }
            zg_vars[name] = da

        ds_zg = xr.Dataset(zg_vars)

        # change variable names for ds_msl
        ds_msl = ds_msl.rename({'msl': 'psl'})
        ds_msl = ds_msl.rename({'10v': 'vas'})
        ds_msl = ds_msl.rename({'10u': 'uas'})

        ds_msl['psl'].attrs.update(
            long_name='Sea level pressure',
            standard_name='air_pressure_at_mean_sea_level',
        )
        ds_msl['vas'].attrs.update(
            long_name='Northward near-surface wind',
        )
        ds_msl['uas'].attrs.update(
            long_name='Eastward near-surface wind',
        )
        # Merge: all 5 variables (msl, u10, v10, zg_a, zg_b) on the same dims
        ds = xr.merge([ds_msl, ds_zg], join='exact')

        ds.to_netcdf(path = final_path, mode = 'w')


/home/b/b382291/.conda/envs/aqua_26/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'latitude', 'longitude') m**2 s**-2
level [300 500]
('time', 'level', 'l

OSError: no files to open

In [11]:
ds.to_netcdf(path = final_path, mode = 'w')


In [ ]:
# removing the time dimension from the topog file 

topog_file_path = '/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/topog.nc'

topog_time = xr.open_dataset(topog_file_path)
topog = topog_time.squeeze("time", drop=True)
topog_file_path = '/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/topog2.nc'

topog.to_netcdf(topog_file_path)

In [6]:


ds_TC = xr.open_dataset(final_path)


In [4]:
ds_TC

<xarray.Dataset> Size: 20GB
Dimensions:  (time: 1460, lat: 451, lon: 771)
Coordinates:
  * time     (time) datetime64[ns] 12kB 2018-01-01 ... 2018-12-31T18:00:00
  * lat      (lat) float64 4kB 45.0 44.9 44.8 44.7 44.6 ... 0.4 0.3 0.2 0.1 0.0
  * lon      (lon) float64 6kB 260.0 260.1 260.2 260.3 ... 336.8 336.9 337.0
Data variables:
    uas      (time, lat, lon) float64 4GB ...
    vas      (time, lat, lon) float64 4GB ...
    psl      (time, lat, lon) float64 4GB ...
    zg300    (time, lat, lon) float64 4GB ...
    zg500    (time, lat, lon) float64 4GB ...
Attributes:
    class:        d1
    stream:       clte
    levtype:      sfc
    type:         fc
    expver:       0001
    date:         20180101
    time:         0
    domain:       g
    Conventions:  CF-1.8
    institution:  ECMWF

In [ ]:

tco2559_phsd = pyhanami.SimulationData(ds_TC, name='tco2559_1x1')

sciskill = pyhanami.ScientificEvaluation(tco2559_phsd)

# Compute TC metrics and related scalar scores for one simulation dataset
tc_analysis= sciskill.compute_tc_scores(
    'tco2559_1x1',
)

tc_analysis.data_cymep.to_netcdf(os.path.join(folder_path, 'TC_results.nc'))

tc_analysis.clim_bias_table(folder_path)
tc_analysis.storm_bias_table(folder_path)
tc_analysis.temp_corr_table(folder_path)
tc_analysis.spatial_corr_table(folder_path)
tc_analysis.save_data(os.path.join(folder_path, 'TC_results_metrics.nc'))
#tc_analysis.linear_plots('/gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/data/tropical_cyclones/plots')
#tc_analysis.spatial_plots('/gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/tropical_cyclones/plots/')

Loaded simulation data 'tco2559_1x1', starting data checks...


1 warnings encountered while checking the provided dataset:
	- Variable 'uas' was missing the corresponding 'long_name' attribute: 'Eastward near-surface wind'. It has been added automatically but better to provide it in the correct format from the beginning.
Data check passed: all variables and coordinates are valid.
Performing TCs analysis for dataset 'tco2559_1x1':
	Years selected for TCs scores computation: 2018-2018.
	IBTrACS TCs data preprocessed and saved to '/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/temp_tracks/ibtracs_v4_1980-2025_10.0_False_1_1.0.txt'.
	Observational TCs data preprocessed and saved to '/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/temp_tracks'.
	Starting simulation data TCs tracking. TempestExtremes output:


/home/b/b382291/git/pyhanami/src/pyhanami/utils/data_general.py:241: UserWarning: As no start year was provided for the TC analysis, the first year available in the 'tco2559_1x1' dataset (2018) will be used.
  warnings.warn(f"As no start year was provided for the {process_name} analysis, the first year available" +
/home/b/b382291/git/pyhanami/src/pyhanami/utils/data_general.py:245: UserWarning: As no end year was provided for the TC analysis, the last year available in the 'tco2559_1x1' dataset (2018) will be used.
  warnings.warn(f"As no end year was provided for the {process_name} analysis, the last year available" +


Arguments:
  --in_data <string> ["/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/temp_tracks/tco2559_1x1_tcs_tempestExtremes_input.nc"] 
  --in_data_list <string> [""] 
  --in_connect <string> [""] 
  --diag_connect <bool> [false] 
  --out <string> ["/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/temp_tracks/detected_nodes_tco2559_1x1.txt"] 
  --out_file_list <string> [""] 
  --searchbymin <string> ["PSL"] (default PSL)
  --searchbymax <string> [""] 
  --searchbythreshold <string> [""] 
  --minlon <double> [0.000000] (degrees)
  --maxlon <double> [0.000000] (degrees)
  --minlat <double> [0.000000] (degrees)
  --maxlat <double> [0.000000] (degrees)
  --minabslat <double> [0.000000] (degrees)
  --mergedist <double> [6.000000] (degrees)
  --mergeequal <bool> [false] 
  --closedcontourcmd <string> ["PSL,200.0,5.5,0;_DIFF(Z300,Z500),-6.0,6.5,1.0"] [var,delta,dist,minmaxdist;...]
  --noclosedcontourcmd <string> [""] [var,delta,dist,minmaxdist;...]
  --thre

FileNotFoundError: [Errno 2] No such file or directory: '/home/b/b382291/git/pyhanami/src/pyhanami/data/tropical_cyclones/temp_tracks/detected_nodes_tco2559_1x1.txt'

In [3]:
import sys, os
print(sys.prefix)                      # the env the kernel is using
print(os.environ["PATH"].split(":")[:5])

/home/b/b382291/.local/share/mamba/envs/pyhanami-env_v0.2.0
['/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/bin', '/sw/spack-levante/git-lfs-2.11.0-oihcwo/bin', '/sw/spack-levante/git-2.43.7-2ofazl/bin', '/home/b/b382291/.local/share/mamba/envs/pyhanami-env_v0.2.0/bin', '/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/condabin']


If you want to start over again to generate data for 2.5 or 0.1 degree, you can use the 10 km experiment on MN5 using the following script to retrieve the data (changing the grid key in the request)

In [ ]:
#!/usr/bin/env python

'''
required fields

- Sea level pressure (psl)
- Eastward and northward wind at 10 m (uas and vas)
- Geopotential height at 300 hPa and 500 hPa (zg300 and zg500)

test experiment: https://gitlab.earth.bsc.es/digital-twins/de_340-2/simulation-runs/-/issues/137

- expid: a3ax
'''

import os
from gsv import GSVRetriever
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

vars = ['10v', '10u', 'msl']

gsv = GSVRetriever()
request = {
  "class": "d1",
  "dataset": "climate-dt",
  "expver": "aaaa",
  "experiment": "hist",
  "activity": "baseline",
  "model": "ifs-nemo",
  "generation": 2,
  "realization": 1,
  "stream": "clte",
  "type": "fc",
  "resolution": "standard",
  "levtype": "sfc",
  "param": vars,
  "date": "19900101/to/19901231",
  "time": "0000/to/2300/by/0600",
  "grid": "0.5/0.5"
}

os.environ['FDB_HOME'] = '/gpfs/scratch/ehpc01/experiments/a3ax/fdb'
request["expver"] = "a3ax"
request["grid"] = "0.5/0.5"
request["resolution"] = "standard"

for year in range(1990,2015,1):
    request["date"] = f"{year}0101/to/{year}1231"
    gsv = GSVRetriever()
    data = gsv.request_data(request)
    data.to_netcdf(f'/gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/data/tropical_cyclones/a3ax_u_v_msl_{year}_tco399_0.5x0.5.nc')
    data.close()
    del data
    print(f"[u,v,msl|year:{year}] -> done")

request['levelist'] = ['300', '500']
request['levtype'] = 'pl'
request['param'] = 'z'
g0 = 9.80665

for year in range(1990,2015,1):
    request["date"] = f"{year}0101/to/{year}1231"
    gsv = GSVRetriever()
    data = gsv.request_data(request)
    data['z'] = data['z'] / g0
    data.to_netcdf(f'/gpfs/scratch/ehpc01/bsc032655/lab/DestinE/debug/scientific-paper-ifs-nemo/data/tropical_cyclones/a3ax_zg_300_500_{year}_tco399_0.5x0.5.nc') 
    data.close()
    del data
    print(f"[z300,z500|year:{year}] -> done")